# NeuraNet — Experimental Validation Lab (Colab)

**Benchmark:** N tâches × 5 workflows × 3 conditions (A/E/F) × M modèles

| | Local / GitHub | Colab (ce notebook) |
|---|---|---|
| Rôle | Code source, API, PostgreSQL dev, tests unitaires | Benchmarks lourds, expériences reproductibles, stats, graphiques |

**Claims testés**
1. Semantic Retrieval (E5 > lexical)
2. Strategy Transfer (E > A)
3. Relevance (E > F shuffled)
4. Zero-context (structurel)
5. Provider neutrality (lift > 0 sur ≥ 2 modèles)
6. Cross-workflow generalization

**Leçons intégrées** (runs locaux précédents):
- Backoff 429-aware (les rate-limits ne sont pas des réponses vides)
- Juge déterministe (temperature=0) + cohérence mesurée
- Réponses vides trackées et exclues avec comptes rapportés
- Workflows balancés, contrôle shuffled à seed fixe, ordre temporel strategy.created_at < target.execution_time

⚠️ **Secrets**: les clés API sont demandées interactivement (`getpass`) — jamais imprimées, jamais sauvegardées dans le notebook.

In [ ]:
# ═══ CELL 1 — CONFIG ═══
import os, time, json, math, random, hashlib, getpass
from datetime import datetime, timedelta
from dataclasses import dataclass, field, asdict
from typing import Optional
import requests

SEED = 42
random.seed(SEED)

# --- API keys (interactive, never logged) ---
GROQ_API_KEY = os.environ.get("GROQ_API_KEY") or getpass.getpass("GROQ_API_KEY: ")

# --- Capability ladder subjects ---
SUBJECTS = [
    {"key": "m7b",  "model": "allam-2-7b",          "params": 7},
    {"key": "m20b", "model": "openai/gpt-oss-20b", "params": 20},
]
JUDGE_MODEL = "openai/gpt-oss-120b"   # jamais subject → pas d'auto-jugement

# --- Scale ---
TASKS_PER_WORKFLOW_TEST = 16   # 16 × 5 = 80 tâches test (N configurable)
TRAIN_PER_WORKFLOW      = 24   # expérience = stratégies du train uniquement
MIN_EFFECT_SIZE         = 0.05
BOOTSTRAP_B             = 5000

WORKFLOWS = ["research", "code", "data", "finance", "decision"]
print(f"Config OK — {TASKS_PER_WORKFLOW_TEST*len(WORKFLOWS)} test tasks, {len(SUBJECTS)} models, judge={JUDGE_MODEL}")

In [ ]:
# ═══ CELL 2 — DATASET (réellement distinct, ordre temporel, split verrouillé) ═══
STOP = {"the","a","an","of","for","and","or","to","in","on","with","using","use","based",
        "from","by","is","are","this","that","it","its","as","at","be","how","what",
        "when","which","their","your"}

ENTITIES = {
 "research": [
  ("banking regulator","Bank of Ghana","ghana"),("energy regulator","NERC Nigeria","nigeria"),
  ("data protection authority","ODPC Kenya","kenya"),("securities commission","SEC Nigeria","nigeria"),
  ("environmental agency","NESREA Nigeria","nigeria"),("telecom authority","NCA Ghana","ghana"),
  ("insurance regulator","NAICOM Nigeria","nigeria"),("pension commission","PenCom Nigeria","nigeria"),
  ("customs authority","GRA Customs Ghana","ghana"),("civil aviation authority","GCAA Ghana","ghana"),
  ("pharmacy council","PC Ghana","ghana"),("central securities depository","CSD Ghana","ghana")],
 "code": [
  ("JWT authentication middleware","Express.js API","token expiry handling"),
  ("rate limiter","REST API gateway","burst traffic handling"),
  ("input validation layer","form submission endpoint","XSS prevention"),
  ("connection pool","PostgreSQL client","connection leak detection"),
  ("file upload handler","multipart form data","malicious file rejection"),
  ("CORS configuration","cross-origin requests","credential exposure"),
  ("session store","Redis-backed sessions","session fixation attack"),
  ("password hashing utility","user registration flow","bcrypt cost factor"),
  ("response cache","read-heavy endpoints","cache invalidation trigger"),
  ("error handler wrapper","async route handlers","unhandled promise rejection"),
  ("audit logging middleware","request trail requirement","PII redaction"),
  ("health check endpoint","kubernetes liveness probe","dependency timeout detection")],
 "data": [
  ("missing value imputation","customer churn dataset","MNAR bias"),
  ("outlier detection","transaction amounts","fraudulent transactions"),
  ("feature scaling","mixed-unit sensor readings","information loss"),
  ("categorical encoding","high-cardinality product IDs","target leakage"),
  ("time series resampling","irregular sensor data","aliasing artifacts"),
  ("duplicate removal","multi-source CRM merge","false positive matches"),
  ("distribution shift detection","A/B test populations","Simpson paradox"),
  ("correlation analysis","macroeconomic indicators","spurious correlation"),
  ("dimensionality reduction","gene expression matrix","variance preservation"),
  ("class imbalance handling","rare disease diagnosis","minority class collapse"),
  ("text normalization","multilingual reviews","diacritic stripping"),
  ("schema validation","ETL ingestion","schema drift detection")],
 "finance": [
  ("Value at Risk calculation","equity portfolio","fat-tailed returns"),
  ("Sharpe ratio optimization","multi-asset allocation","risk-free rate selection"),
  ("bond duration matching","liability stream","yield curve inversion"),
  ("options pricing model","European call options","volatility smile"),
  ("portfolio rebalancing","60/40 stock-bond mix","transaction costs"),
  ("credit risk scoring","SME loan applicants","default correlation"),
  ("currency hedging strategy","export revenue exposure","basis risk"),
  ("DCF valuation","early-stage startup","terminal value assumption"),
  ("Monte Carlo simulation","retirement planning","sequence of returns risk"),
  ("stress testing","banking balance sheet","liquidity crunch scenario"),
  ("ESG screening","institutional fund mandate","greenwashing detection"),
  ("factor decomposition","hedge fund returns","style drift attribution")],
 "decision": [
  ("cloud provider selection","cost reliability compliance","vendor lock-in risk"),
  ("database technology choice","SQL vs NoSQL tradeoffs","consistency requirements"),
  ("make-vs-buy analysis","internal tool vs SaaS","maintenance burden"),
  ("programming language selection","team skills ecosystem","long-term support"),
  ("architecture pattern choice","monolith vs microservices","organizational maturity"),
  ("vendor negotiation","contract renewal pricing","switching costs"),
  ("feature prioritization","limited engineering capacity","opportunity cost"),
  ("market entry timing","first-mover advantage","regulatory uncertainty"),
  ("hiring strategy","senior specialist vs generalist","culture fit assessment"),
  ("security investment level","threat model alignment","compliance mandate overlap"),
  ("partnership structure","revenue share vs equity","exit clause fairness"),
  ("geographic expansion","adjacent market entry","localization complexity")],
}

TEMPLATES = {
 "research": [
  ("Identify the official {ent} of {jur}.","Search the official government portal for the {ent} in {jur}, verify domain authenticity, cross-check the finding with a second independent source."),
  ("Find the enforcement powers of the {ent} in {jur}.","Locate the enabling act creating the {ent} in {jur}, extract the enforcement sections, cite the specific legal provisions."),
  ("Determine the current leadership of the {ent} in {jur}.","Check the official leadership page of the {ent} in {jur}, verify against recent press announcements before concluding."),
  ("Summarize recent regulatory actions by the {ent}.","Search official press releases from the {ent}, filter to the last twelve months, summarize each enforcement action with dates.")],
 "code": [
  ("Implement {ent} for a project involving {ctx}.","Analyze the requirements for the {ent}, address {risk} explicitly, implement, then add unit tests covering both success and failure paths."),
  ("Debug a failing {ent} in production.","Reproduce the failure locally, add diagnostic logging around the {ent}, isolate the root cause related to {risk}, fix, add a regression test."),
  ("Write integration tests for {ent}.","Set up an isolated test environment, create realistic fixtures for the {ent}, mock externals, cover {risk} scenarios end to end."),
  ("Refactor existing code to improve {ent}.","Identify weaknesses in the current {ent}, extract reusable components while preserving behavior, update tests including {risk} coverage."),
  ("Document the security implications of {ent}.","Threat-model the {ent}, enumerate attacks targeting {risk}, document mitigations with concrete configuration examples.")],
 "data": [
  ("Apply {ent} to a dataset containing {ctx}.","Profile column distributions first, choose the technique accounting for {risk}, validate results statistically, document assumptions."),
  ("Diagnose anomalies when performing {ent}.","Visualize raw distributions, compute summary statistics, flag observations violating expectations related to {risk}."),
  ("Build a reproducible pipeline for {ent}.","Define a schema contract, implement idempotent transforms, add unit tests covering {risk} edge cases."),
  ("Compare two approaches to {ent}.","Define one evaluation metric up front, run both methods on identical data, measure downstream impact regarding {risk}."),
  ("Document pitfalls of {ent} with {ctx}.","List common failure modes, explain how {risk} manifests, provide detection heuristics and corrective actions.")],
 "finance": [
  ("Perform {ent} for {ctx}, accounting for {risk}.","Gather historical inputs, define the mathematical model, handle {risk} explicitly, compute, sanity-check against published benchmarks."),
  ("Evaluate whether {ent} is appropriate here.","State the assumptions behind {ent}, test whether {risk} violates them, recommend an alternative when they fail."),
  ("Build a spreadsheet model for {ent}.","Separate inputs, formulas and outputs clearly, implement {ent} mechanics, add sensitivity analysis around {risk}."),
  ("Explain limitations of {ent} to a non-technical stakeholder.","Use concrete analogies without jargon, highlight how {risk} breaks the model, give one real historical example."),
  ("Audit an existing {ent} calculation.","Trace every input to its source, recompute independently, reconcile differences, flag exposure to {risk}.")],
 "decision": [
  ("Decide on {ent} considering {crit}.","Define weighted criteria covering {crit}, score every option, run sensitivity analysis, document the rationale and reversibility."),
  ("Evaluate alternatives for {ent}.","List viable options, build a decision matrix including {crit}, eliminate dominated choices, inspect the Pareto frontier."),
  ("Justify a recommendation about {ent}.","Assemble evidence for the position, address counterarguments about {crit}, present a structured written argument."),
  ("Reverse-engineer a surprising choice about {ent}.","Infer hidden constraints beyond stated {crit} from observed behavior, avoid hindsight bias, state assumptions."),
  ("Design a review checklist for {ent} decisions.","Extract recurring criteria like {crit}, define pass/fail gates, pilot the checklist on past decisions.")],
}

WRONG_STRATEGY = {
 "research":"Search entertainment news and celebrity gossip sites.",
 "code":"Delete all tests and disable type checking to ship faster.",
 "data":"Randomly shuffle rows and drop columns without inspection.",
 "finance":"Extrapolate linearly from last week's price movement.",
 "decision":"Flip a coin without evaluating any criterion.",
}

def build_dataset():
    now = datetime.utcnow()
    train, test, strategies = [], [], []
    tid = 0
    for wf in WORKFLOWS:
        ents = ENTITIES[wf]; tpls = TEMPLATES[wf]
        # interleaved round-robin → chaque workflow fournit exactement ses quotas
        combos = [(e,t) for t_i,t in enumerate(tpls) for e_i,e in enumerate(ents)]
        random.shuffle(combos)
        n_train, n_test = TRAIN_PER_WORKFLOW, TASKS_PER_WORKFLOW_TEST
        picks = combos[:n_train+n_test]
        for i,(e,t) in enumerate(picks):
            ent,ctx,jur_or_risk = e[0], e[1], e[2]
            is_train = i < n_train
            created = now - timedelta(hours=(10000 - tid))
            executed = now - timedelta(hours=(1000 - tid))
            rec = {
                "id": f"{wf[:1].upper()}{tid:03d}", "workflow": wf,
                "created_at": created.isoformat()+"Z", "execution_time": executed.isoformat()+"Z",
            }
            tpl_task, tpl_strat = (t[0], t[1])
            rec["task"] = tpl_task.format(ent=ent, ctx=ctx, risk=jur_or_risk, crit=ctx, jur=jur_or_risk)
            bucket = train if is_train else test
            bucket.append(rec)
            if is_train:
                strategies.append({
                    "id": f"S_{rec['id']}", "source_task_id": rec["id"], "workflow": wf,
                    "created_at": rec["created_at"],
                    "strategy_text": tpl_strat.format(ent=ent, risk=jur_or_risk, jur=jur_or_risk),
                })
            tid += 1
    return train, test, strategies

TRAIN, TEST, STRATEGIES = build_dataset()
temporal_leakage = sum(1 for s in STRATEGIES
                       if s["created_at"] >= next(t["execution_time"] for t in TEST+TRAIN if t["id"]==s["source_task_id"]))
print(f"Train={len(TRAIN)} Test={len(TEST)} Strategies={len(STRATEGIES)} Temporal_leakage={temporal_leakage}")
assert temporal_leakage == 0

In [ ]:
# ═══ CELL 3 — E5 EMBEDDINGS (GPU) + RETRIEVAL + HARD FILTERS ═══
!pip -q install sentence-transformers
from sentence_transformers import SentenceTransformer
import numpy as np

device = "cuda" if __import__("torch").cuda.is_available() else "cpu"
print("E5 device:", device)
E5 = SentenceTransformer("intfloat/multilingual-e5-small", device=device)  # 384 dims

STRAT_EMB = E5.encode([f"passage: {s['strategy_text']}" for s in STRATEGIES],
                      batch_size=128, normalize_embeddings=True, show_progress_bar=True)

WF_COMPAT = {"research":{"research"},"code":{"code"},"data":{"data"},
             "finance":{"finance"},"decision":{"decision","research","data"}}

def content_words(text):
    import re
    return [w for w in re.sub(r"[^a-z\s]"," ",text.lower()).split()
            if len(w)>3 and w not in STOP]

def entity_overlap(task, strat):
    t=set(content_words(task)); s=set(content_words(strat))
    return len(t & s)/max(len(t),1)

def retrieve(task, wf, top_k=5):
    q = E5.encode([f"query: {task}"], normalize_embeddings=True)[0]
    sims = STRAT_EMB @ q
    ranked = np.argsort(-sims)
    kept = [(i, float(sims[i])) for i in ranked
            if WF_COMPAT[wf] & {STRATEGIES[i]["workflow"]}
            and entity_overlap(task, STRATEGIES[i]["strategy_text"]) >= 0.12]
    return kept[:top_k]

def shuffled_strategy(task_idx):
    rng = random.Random(f"{SEED}:{task_idx}")   # contrôle shuffled reproductible
    return STRATEGIES[rng.randrange(len(STRATEGIES))]

In [ ]:
# ═══ CELL 4 — LLM CLIENTS (429-aware) + JUGE DÉTERMINISTE ═══
GROQ_URL = "https://api.groq.com/openai/v1/chat/completions"
MODEL_MAX_TOKENS = {"allam-2-7b":1100, "openai/gpt-oss-20b":3500,
                    "qwen/qwen3.6-27b":2000, "openai/gpt-oss-120b":512}

def _strip_think(text):
    import re
    return re.sub(r"<think>[\s\S]*?</think>", "", text).strip()

def llm(model, messages, temperature=0.7, max_retries=6):
    delays = [3,8,15,30,45,60]
    budget = MODEL_MAX_TOKENS.get(model, 1500)
    for attempt,wait in enumerate(delays[:max_retries]):
        try:
            r = requests.post(GROQ_URL,
                headers={"Authorization":f"Bearer {GROQ_API_KEY}"},
                json={"model":model,"messages":messages,"temperature":temperature,
                      "max_tokens":budget}, timeout=90)
            if r.status_code == 429:
                time.sleep(wait); continue
            r.raise_for_status()
            c = _strip_think(r.json()["choices"][0]["message"]["content"] or "")
            if not c and attempt>=2:
                budget = min(budget*2, 8000)   # reasoning a mangé le budget
            if c:
                u = r.json().get("usage",{})
                return {"content":c, "tokens":u.get("prompt_tokens",0)+u.get("completion_tokens",0)}
            time.sleep(wait)
        except requests.RequestException:
            time.sleep(wait)
    return {"content":"", "tokens":0}   # EMPTY — tracké par l'appelant

def judge_quality(task, output):
    """Juge aveugle déterministe. Fallback heuristique si quota indisponible."""
    if len(output or "") < 30:
        return 0.10, True
    for attempt,wait in enumerate([5,20,30,45]):
        try:
            r = requests.post(GROQ_URL,
                headers={"Authorization":f"Bearer {GROQ_API_KEY}"},
                json={"model":JUDGE_MODEL,"temperature":0,"max_tokens":MODEL_MAX_TOKENS[JUDGE_MODEL],
                      "messages":[{"role":"user","content":
                          f"Score this answer from 0.0 to 1.0 on correctness, completeness, actionability for the task. End your reply with the number on the last line.\n\nTASK: {task}\n\nANSWER: {output[:600]}"}]},
                timeout=90)
            if r.status_code == 429:
                time.sleep(wait); continue
            r.raise_for_status()
            import re
            nums = re.findall(r"([01](?:\.\d+)?)", _strip_think(r.json()["choices"][0]["message"]["content"]))
            if nums:
                time.sleep(3)  # lissage TPM
                return min(1.0,max(0.0,float(nums[-1]))), False
        except requests.RequestException:
            time.sleep(wait)
    return heuristic_quality(output,task), True

def heuristic_quality(output, task):
    if len(output or "")<50: return 0.10
    words = set(content_words(task))
    lo = output.lower()
    score = 0.25*min(len(output)/800,1) \
          + 0.20*(1 if any(x in output for x in ["\n\n","\n- ","\n1.","\n*"]) else 0) \
          + 0.25*(1 if any(w.isupper() and len(w)>2 for w in output.split()) else 0) \
          + 0.30*(len(words & set(content_words(output)))/max(len(words),1))
    return round(score,3)

In [ ]:
# ═══ CELL 5 — RUNNER (conditions A/E/F × modèles ; sources identiques entre modèles) ═══
results = []
empty_counts = {s["key"]:0 for s in SUBJECTS}
judge_fallbacks = 0

def search_web(query, k=3):
    """Optionnel: brancher Tavily ici. Sans clé, retourne [] et E≈A pour cette tâche
    (le run reste valide pour la comparaison INTER-modèles puisque les sources sont partagées)."""
    return []

def source_block(rs):
    if not rs: return ""
    lines = [f"[{i+1}] {x.get('title','')}: {str(x.get('snippet',''))[:150]} ({x.get('url','')})"
             for i,x in enumerate(rs)]
    return "\n\nSOURCES:\n" + "\n".join(lines)[:1100]

for idx, task in enumerate(TEST):
    row = {"taskId":task["id"], "workflow":task["workflow"], "models":{}, "retrieval":{}}
    kept = retrieve(task["task"], task["workflow"])
    best_sim = kept[0][1] if kept else None
    best_strat = STRATEGIES[kept[0][0]] if kept else None
    shuf = shuffled_strategy(idx)
    row["retrieval"] = {"bestSim":best_sim, "bestStratId":best_strat["id"] if best_strat else None,
                        "shuffledId":shuf["id"], "poolSize":len(kept)}
    # requêtes de recherche DÉTERMINISTES par tâche → mêmes sources pour tous les modèles
    kw = " ".join(content_words(best_strat["strategy_text"])[:4]) if best_strat else ""
    e_query = (task["task"][:120]+" "+kw)[:180]
    kw_f = " ".join(content_words(shuf["strategy_text"])[:4])
    f_query = (task["task"][:120]+" "+kw_f)[:180]
    e_sources, f_sources = search_web(e_query), search_web(f_query)

    for subj in SUBJECTS:
        sys_plain  = "You are a helpful assistant."
        sys_search = sys_plain + " Answer using the provided sources when relevant. Cite [n]."
        outs = {}
        outs["A"] = llm(subj["model"], [{"role":"system","content":sys_plain},{"role":"user","content":task["task"]}])
        outs["E"] = llm(subj["model"], [{"role":"system","content":sys_search},{"role":"user","content":task["task"]+source_block(e_sources)}])
        outs["F"] = llm(subj["model"], [{"role":"system","content":sys_search},{"role":"user","content":task["task"]+source_block(f_sources)}])
        m = {"params":subj["params"]}
        for cond,out in outs.items():
            if not out["content"]:
                empty_counts[subj["key"]] += 1
                m[cond] = {"q":None, "tokens":out["tokens"]}
            else:
                q,fb = judge_quality(task["task"], out["content"])
                if fb: judge_fallbacks += 1
                m[cond] = {"q":q, "tokens":out["tokens"]}
        m["liftEA"] = round(m["E"]["q"]-m["A"]["q"],3) if None not in (m["E"]["q"],m["A"]["q"]) else None
        m["liftEF"] = round(m["E"]["q"]-m["F"]["q"],3) if None not in (m["E"]["q"],m["F"]["q"]) else None
        row["models"][subj["key"]] = m
    results.append(row)
    done = idx+1
    if done % 5 == 0 or done==len(TEST):
        print(f"[{done}/{len(TEST)}] {task['id']} ({task['workflow']}) " +
              " ".join(f"{s['key']}:Δ={row['models'][s['key']]['liftEA']}" for s in SUBJECTS))

print("\nEmpty responses:", empty_counts, "| Judge fallbacks:", judge_fallbacks)

In [ ]:
# ═══ CELL 6 — STATISTIQUES (bootstrap CI95 apparié, Cohen's d, taux de transfert) ═══
import numpy as np

def paired_stats(diffs):
    diffs = [d for d in diffs if d is not None]
    n=len(diffs)
    if n<3: return {"n":n,"note":"insufficient"}
    a=np.array(diffs); rng=np.random.default_rng(SEED)
    boots=np.array([rng.choice(a,n,replace=True).mean() for _ in range(BOOTSTRAP_B)])
    ci=np.percentile(boots,[2.5,97.5]); mean=a.mean(); sd=a.std(ddof=1)
    return {"n":n,"mean":round(float(mean),4),"median":round(float(np.median(a)),4),
            "sd":round(float(sd),4),"ci95":[round(float(ci[0]),4),round(float(ci[1]),4)],
            "cohensD":round(float(mean/(sd/math.sqrt(n))) if sd>0 else 0.0,3),
            "significant":bool(ci[0]>0 or ci[1]<0),
            "posRate":round(float((a> MIN_EFFECT_SIZE).mean()),3),
            "negRate":round(float((a<-MIN_EFFECT_SIZE).mean()),3)}

analysis={"meta":{
    "generated_at":datetime.utcnow().isoformat()+"Z", "seed":SEED,
    "test_tasks":len(TEST), "train_tasks":len(TRAIN),
    "min_effect_size":MIN_EFFECT_SIZE, "bootstrap_b":BOOTSTRAP_B,
    "subjects":[{k:s[k] for k in ('key','model','params')} for s in SUBJECTS],
    "judge":JUDGE_MODEL, "empty_responses":empty_counts, "judge_fallbacks":judge_fallbacks},
  "byModel":{}, "byWorkflow":{}}

for s in SUBJECTS:
    k=s["key"]
    analysis["byModel"][k]={
        "EA":paired_stats([r["models"][k]["liftEA"] for r in results]),
        "EF":paired_stats([r["models"][k]["liftEF"] for r in results]),
        "baselineQ":round(float(np.mean([r["models"][k]["A"]["q"] for r in results
                              if r["models"][k]["A"]["q"] is not None])),3)}

for wf in WORKFLOWS:
    sub=[r for r in results if r["workflow"]==wf]
    analysis["byWorkflow"][wf]={
        s["key"]:paired_stats([r["models"][s["key"]]["liftEA"] for r in sub]) for s in SUBJECTS}

print(json.dumps(analysis["byModel"], indent=2))
print("\nCapability axis — baselineQ (doit croître avec params):",
      {k:v["baselineQ"] for k,v in analysis["byModel"].items()})
json.dump({"analysis":analysis,"results":results}, open("ladder_results.json","w"), indent=2)

In [ ]:
# ═══ CELL 7 — GRAPHIQUES ═══
import matplotlib.pyplot as plt

keys=[s["key"] for s in SUBJECTS]; labels=[f"{s['params']}B" for s in SUBJECTS]
meansEA=[analysis["byModel"][k]["EA"].get("mean") for k in keys]
cisEA  =[analysis["byModel"][k]["EA"].get("ci95",[None,None]) for k in keys]

fig,axes=plt.subplots(1,2,figsize=(13,4.5))
ax=axes[0]
ax.errorbar(range(len(keys)), meansEA,
            yerr=[[m-(c[0] if c else m) for m,c in zip(meansEA,cisEA)],
                  [(c[1] if c else m)-m for m,c in zip(meansEA,cisEA)]],
            marker="o", capsize=5)
ax.axhline(0,color="gray",lw=0.8); ax.axhline(MIN_EFFECT_SIZE,color="red",ls="--",lw=0.8,label=f"min effect +{MIN_EFFECT_SIZE}")
ax.set_xticks(range(len(keys))); ax.set_xticklabels(labels)
ax.set_xlabel("Model capability (params)"); ax.set_ylabel("Transfer lift E−A")
ax.set_title("NeuraNet gain vs model capability"); ax.legend()

ax=axes[1]
width=0.35; xs=np.arange(len(WORKFLOWS))
for i,k in enumerate(keys):
    vals=[[analysis["byWorkflow"][wf][k].get("mean",0)] for wf in WORKFLOWS]
    ax.bar(xs+i*width, [v[0] or 0 for v in vals], width, label=k)
ax.set_xticks(xs+width/2); ax.set_xticklabels(WORKFLOWS, rotation=20)
ax.axhline(0,color="gray",lw=0.8); ax.set_ylabel("Lift E−A")
ax.set_title("Lift by workflow"); ax.legend()
plt.tight_layout(); plt.savefig("ladder_plots.png",dpi=140); plt.show()

In [ ]:
# ═══ CELL 8 — RAPPORT MARKDOWN + EXPORT ═══
def fmt(st): return (f"{st['mean']:+.3f} CI[{st['ci95'][0]:+.3f},{st['ci95'][1]:+.3f}] "
                     f"d={st['cohensD']} sig={st['significant']} pos={st['posRate']} neg={st['negRate']}"
                     if 'mean' in st else str(st))

lines=["# NeuraNet Capability-Ladder Report (Colab)","",
       f"Generated: {analysis['meta']['generated_at']} | Seed {SEED} | "
       f"Test N={len(TEST)} | Judge `{JUDGE_MODEL}` | Empty: {empty_counts}","",
       "## Transfer lift (E−A) par modèle","",
       "| Model | Stats |","|---|---|"]
for s in SUBJECTS:
    lines += [f"| {s['params']}B ({s['model']}) | {fmt(analysis['byModel'][s['key']]['EA'])} |"]
lines += ["", "## Relevance (E−F)","","| Model | Stats |","|---|---|"]
for s in SUBJECTS:
    lines += [f"| {s['params']}B | {fmt(analysis['byModel'][s['key']]['EF'])} |"]
lines += ["", "## Par workflow","","| Workflow | "+" | ".join(labels)+" |","|---|"+"---|"*len(keys)]
for wf in WORKFLOWS:
    lines.append("| "+wf+" | "+" | ".join(
        str(analysis['byWorkflow'][wf][k].get('mean','—')) for k in keys)+" |")
lines += ["","> Interprétation: si lift décroît avec params → NeuraNet = guidance surtout pour agents légers.",
          "> NE PAS conclure au-delà des données. Statut: PRELIMINARY sauf IC excluant 0 sur plusieurs modèles."]
report="\n".join(lines)
open("CAPABILITY_LADDER_REPORT.md","w").write(report)
print(report)
from google.colab import files
files.download("ladder_results.json"); files.download("CAPABILITY_LADDER_REPORT.md"); files.download("ladder_plots.png")